# Floating Point and Root Stability

In [ ]:
using Plots

These are the same polynomial, one is just expanded:

In [ ]:
p1(x) = x^4 - 4x^3 + 6x^2 - 4x + 1
p2(x) = (x-1)^4

Only one root at $x=1$, with multiplicity 4.

In [ ]:
xx = LinRange(0, 2, 100)
plot(xx, p1.(xx), label="Expanded", xlabel="x", ylabel="y", title="Plot of x^4 - 4x^3 + 6x^2 - 4x + 1")
plot!(xx, p2.(xx), label="Factored")

Zoom in:

In [ ]:
# xx = LinRange(.995, 1.005, 100); # defaults to double precision
# xx = Float32.(LinRange(.995, 1.005, 100)); # switch to single precision
#xx = LinRange(.9995, 1.0005, 100) # zoom in with double precision
xx = LinRange(.999995, 1.000005, 100) # zoom in some more
plot(xx, p1.(xx), label="Expanded", xlabel="x", ylabel="y", title="Plot of x^4 - 4x^3 + 6x^2 - 4x + 1")
plot!(xx, p2.(xx), label="Factored")

We have ``lost'' the root to single precision arithmetic in the expanded form.  It's also a problem with double precsision when we zoome in enough.

# Bisection Method

## Example 1

In [ ]:
f(x) = cos(x) - x;

xx = LinRange(0,2,200)
plot(xx, f.(xx), label="cos(x) - x", xlabel="x", ylabel="y")
plot!(xx, 0. * xx, label="y = 0")

Run bisection method on this with initial interval $[0,2]$:

In [ ]:
# set the initial interval for the bisection method
a = 0;
b = 2;
nmax = 10;
for n in 1:nmax
    # compute the midpoint of the current interval
    c = (a + b) / 2;
    # determine which interval we are in and update [a,b]
    if f(a) * f(c) < 0
        b = c
    else
        a = c
    end
    println("Iteration $n: a = $a, b = $b, c = $c, f(c) = $(f(c))")
end




We want to package up bisection method as a function:

Arguments after the `;` are named and optional.  If you do not pass them, they use the default, indicated value.

In [ ]:
function bisection_method(f, a, b; nmax=100, δ = 1e-6, ϵ = 1e-6)
    if (f(a) * f(b) > 0)
        # throws an error and prints the message, and it terminates the function
        error("The function must have opposite signs at the endpoints of the interval.")
    end

    # need to include this to enusre we are always in scope
    c = 0;

    for n in 1:nmax
        half_width = (b-a)/2;
        c = a + half_width; # compute the midpoint
        # check our result
        if abs(f(c)) < ϵ || half_width < δ
            # print diagnostic information and return
            # @info("Root found at x = $c after $n iterations")
            if abs(f(c)) < ϵ
                # f(c) close enough to zero
                @info("Root found at x = $c after $n iterations based on function value")
            elseif half_width < δ
                # (b-a)/2 small enough
                @info("Root found at x = $c after $n iterations based on interval width")
            end
            return c;
        # otherwise proceed to the next iteration
        elseif f(a)*f(c) < 0
            b = c;
        else
            a = c
        end
    end
    # prints an error message, but does not break the function
    @error("$nmax iterations reached without finding a root.")
    return c
end

In [ ]:
c = bisection_method(f, 0, 2);
@show f(c);

In [ ]:

xx = LinRange(0,2,200)
plot(xx, f.(xx), label="cos(x) - x", xlabel="x", ylabel="y")
plot!(xx, 0. * xx, label="y = 0")
scatter!([c], [f(c)], label="Root")

Better estimates?  Use more stringent `δ` and `ϵ`

In [ ]:
c = bisection_method(f, 0, 2, ϵ=1e-8); #stopped for a different reason
@show f(c);

In [ ]:
c = bisection_method(f, 0, 2,δ=1e-10, ϵ=1e-8); #stopped for a different reason
@show c;
@show f(c);

The following has `nmax` too low and will return error because it can't satisfy the stopping criterion.

In [ ]:
c = bisection_method(f, 0, 2,δ=1e-10, ϵ=1e-8, nmax=10); #stopped for a different reason
@show c;
@show f(c);

In [ ]:
bisection_method(f, 0,0.5)

## Example 2

In [ ]:
f(x) = tan(x) - x;
xx = LinRange(-3*π, 3*π,10000);
plot(xx, f.(xx),xlabel="x", ylabel="f(x)",label="")
plot!(xx, 0. *xx, label="y=0")
ylims!(-10,10)

In [ ]:
@show π/2;

Get the first root

In [ ]:
bisection_method(f, -1, 1)

Next positive root is in $(\pi/2, 3\pi/2)$

In [ ]:
@show π/2;
@show 3π/2;

In [ ]:
c = bisection_method(f, 1.6, 4.7);
@show f(c);

In [ ]:
xx = LinRange(-3*π, 3*π,10000);
plot(xx, f.(xx),xlabel="x", ylabel="f(x)",label="")
plot!(xx, 0. *xx, label="y=0")
scatter!([c], [f(c)], label="root")
ylims!(-10,10)

Next positive root is in $(3\pi/2, 5\pi/2)$

In [ ]:
@show 3π/2 # need a a little bit bigger than this
@show 5π/2 # need a b little bit smaller than this

In [ ]:
c = bisection_method(f, 4.8, 7.8);
@show f(c);

xx = LinRange(-3*π, 3*π,10000);
plot(xx, f.(xx),xlabel="x", ylabel="f(x)",label="")
plot!(xx, 0. *xx, label="y=0")
scatter!([c], [f(c)], label="root")
ylims!(-10,10)